[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lowdanie/hartree-fock-solver/blob/main/notebooks/pyscf_bench.ipynb)

# PySCF Benchmark

This notebook compares the total energy computed by `slaterform` to [PySCF](https://pyscf.org/).

## System Setup

Run the following cells to initialize the notebook.

In [ ]:
# @title Pip Installs

!pip install -qq pyscf
!pip install -qq git+https://github.com/lowdanie/hartree-fock-solver

In [ ]:
!pip freeze | grep slaterform

In [ ]:
# @title Imports { display-mode: "form" }

import jax

jax.config.update("jax_enable_x64", True)

import dataclasses
import io
import time
from typing import Callable, NamedTuple
from collections.abc import Sequence
import functools

import jax.numpy as jnp
import numpy as np
import optax

import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

import pyscf
import pubchempy as pcp

import slaterform as sf
import slaterform.hartree_fock.scf as scf

In [ ]:
print(f"JAX Backend: {jax.devices()[0]}")

In [ ]:
# @title Adapters { display-mode: "form" }


def load_molecule(name: str) -> scf.Molecule:
    compounds = pcp.get_compounds(name, "name", record_type="3d")
    atoms = sf.adapters.pubchem.load_geometry(compounds[0])
    return sf.Molecule.from_geometry(atoms, basis_name="sto-3g")


def build_pyscf_molecule(molecule: sf.Molecule) -> pyscf.gto.Mole:
    numbers = [atom.number for atom in molecule.atoms]
    positions = [tuple(atom.position) for atom in molecule.atoms]
    return pyscf.gto.M(
        atom=list(zip(numbers, positions)),
        basis="sto-3g",
        unit="Bohr",
        symmetry=False,
    )

## Energy Evaluation

Select a molecule and compare energies computed by slaterform and pyscf.

In [ ]:
# Select a molecule
MOLECULE_NAME = "Aspirin"  # @param {type:"string"}

In [ ]:
PRECISION = "f64"  # @param ["f32", "f64"]

In [ ]:
# @title Load Geometry
print(f"Loading geometry for: {MOLECULE_NAME}")
molecule = load_molecule(MOLECULE_NAME)

In [ ]:
# @title Compute Energy With Slaterform

print(f"Computing energy with precision: {PRECISION}")
integral_dtype = jnp.float32 if PRECISION == "f32" else jnp.float64

basis = sf.BatchedBasis.from_molecule(
    molecule, batch_size_1e=128, batch_size_2e=256
)
options = scf.Options(
    solver=sf.fixed_point.AndersonParams(max_iter=50, tol=1e-8, m=5, beta=0.7),
    integral_strategy=scf.CachedStrategy(dtype=integral_dtype),
    perturbation=1e-10,
)
solve_fn = jax.jit(scf.solve)
result = solve_fn(basis, options)

In [ ]:
# @title Compute Energy With Pyscf

molecule_pyscf = build_pyscf_molecule(molecule)
mf = pyscf.scf.RHF(molecule_pyscf)
mf.verbose = 0
mf.kernel()

In [ ]:
# @title Result

print(f"{MOLECULE_NAME} with slaterform precision {PRECISION}")
print(f"Slaterform Energy: {result.total_energy:.10f}")
print(f"Pyscf Energy:      {mf.e_tot:.10f}")
print(f"Absolute Error:    {abs(result.total_energy - mf.e_tot):.10f}")